# Import lIbraries

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Fake Dataset

In [4]:
fake_df = pd.read_excel(
    r"C:\Users\Admin\Downloads\PolyglotFakeFacts A multilingual dataset of fake a\Fake.xlsx"
)

print("Fake Dataset Shape:", fake_df.shape)
fake_df.head()

Fake Dataset Shape: (4912, 10)


,Gathering date,News date,Language,URL,Domain,Keywords,News headline,News original text,English translated version,Label
0,2023-05-30,2019-10-22,arabic,https://arabic.sputniknews.com/russia/20191022...,arabic.sputniknews.com,Crimea,"موسكو تكشف عن وجود مشروع سياسي للغرب في دولة ""...",كشف المتحدث باسم الرئاسة الروسية دميتري بيسكوف...,Russian presidential spokesman Dmitry Peskov r...,fake
1,2023-05-30,2019-07-18,arabic,https://arabic.rt.com/middle_east/1032932-%D8%...,arabic.rt.com,"Idlib, Syrian War",أوكرانيا: مستعدون للحوار مع روسيا حول القرم,أحمــــــــــــــــــــد عمرخالـــــــــــــــ...,Ahmed Omar Khaled NajehUkrainian Foreign Minis...,fake
2,2023-05-30,2019-11-27,arabic,https://arabic.sputniknews.com/russia/20191127...,sptnkne.ws,North Africa,مرصد الزلازل الأردني: المنطقة تشهد هزات أرضية ...,مرصد الزلازل الأردنيقال مدير مرصد الزلازل الأر...,"The Jordanian Seismological Observatory, Ghass...",fake
3,2023-05-30,2020-06-01,arabic,https://arabic.rt.com/world/1119959-%D8%A7%D9%...,arabic.rt.com,"Diplomacy with Russia, Anti-Russian, Elections",العقوبات ضد روسيا أبدية ويجب على موسكو التصرف ...,أحمــــــــــــــــــــد عمرخالـــــــــــــــ...,Ahmed Omar Khaled NajehThe Russian Deputy Fore...,fake
4,2023-05-30,2020-04-11,arabic,https://russiarab.com/archives/39296/,Russia in Arabic,"EU budget, coronavirus, EU disintegration","الكرملين: اتهامات التشيك لروسيا ""مشينة """,أحمــــــــــــــــــــد عمرخالـــــــــــــــ...,Ahmed Omar Khaled Nageh The Russian presidenti...,fake


# Load Real Dataset

In [6]:
real_df = pd.read_excel(
    r"C:\Users\Admin\Downloads\PolyglotFakeFacts A multilingual dataset of fake a\Real.xlsx"
)

print("Real Dataset Shape:", real_df.shape)
real_df.head()

Real Dataset Shape: (5294, 10)


,gathering date,news date,url,domain,language,keywords,news headline,news original text,english translated version,label
0,2023-06-08 16:35:39.788000,NaN,https://www.dnes.bg/izbori-2023/2023/06/07/iav...,www.dnes.bg,Bulgarian,"politics, academician, nikolay, voted",Явор Божанков: Вчера ни атакуваха брутално,С абсолютна убеденост гласувах за наш кабинет ...,"With absolute conviction, I voted for our cabi...",real
1,2023-06-08 16:05:45.672000,NaN,https://www.dnes.bg/politika/2023/06/08/shte-s...,www.dnes.bg,Bulgarian,"politics, yavor, bozhankov, twice","Ще спрат ли скандалите в НС? 'Възраждане'"" дад...","След като вчера на два пъти депутатите от ''""В...","After twice yesterday the MPs from ""Vazrazhdan...",real
2,2023-06-08 16:47:54.618000,NaN,https://www.dnes.bg/mish-mash/2023/06/04/short...,www.dnes.bg,Bulgarian,"sports, summer, garden, close",Шорти от деним - неизменна част от летния гард...,"Лятото е толкова близо, че на практика можем д...",Summer is so close we can practically taste it...,real
3,2023-06-08 16:06:00.282000,NaN,https://www.dnes.bg/temida/2023/06/08/shestima...,www.dnes.bg,Bulgarian,"legal, geshev's, ognyan, officers",Шестимата от ВСС оттеглиха искането за оставка...,Заседанието на ВСС за оставка на Гешев се прео...,The session of the SJC for Geshev's resignatio...,real
4,2023-06-08 16:53:25.269000,NaN,https://www.dnes.bg/akoshtete-vqrvaite/2020/08...,www.dnes.bg,Bulgarian,"politics, sunny, beach, party",Шампанско и... ковчег! Кръст и монахини на пар...,Парти на плаж в Слънчев бряг предизвика стотиц...,A beach party in Sunny Beach caused hundreds o...,real


# Create Target Labels

In [8]:
fake_df["target"] = 1
real_df["target"] = 0

print(fake_df["target"].value_counts())
print(real_df["target"].value_counts())

target
1    4912
Name: count, dtype: int64
target
0    5294
Name: count, dtype: int64


# Rename Text Columns

In [10]:
fake_df = fake_df.rename(
    columns={"English translated version": "text"}
)

real_df = real_df.rename(
    columns={"english translated version": "text"}
)

# Keep Required Columns

In [12]:
fake_df = fake_df[["text", "target"]]
real_df = real_df[["text", "target"]]

# Combine Datasets

In [14]:
df = pd.concat([fake_df, real_df], ignore_index=True)

print("Combined Shape:", df.shape)

print(df["target"].value_counts())

Combined Shape: (10206, 2)
target
0    5294
1    4912
Name: count, dtype: int64


# Remove Missing Values

In [16]:
df.dropna(inplace=True)

print(df.shape)

print(df["target"].value_counts())

(10130, 2)
target
0    5268
1    4862
Name: count, dtype: int64


# Text Cleaning Function

In [18]:
ps = PorterStemmer()

def clean_text(text):

    text = str(text)

    text = re.sub('[^a-zA-Z]', ' ', text)

    text = text.lower()

    words = text.split()

    words = [
        ps.stem(word)
        for word in words
        if word not in stopwords.words('english')
    ]

    return " ".join(words)

# Apply Cleaning

In [20]:
df["cleaned_text"] = df["text"].apply(clean_text)

df.head()

,text,target,cleaned_text
0,Russian presidential spokesman Dmitry Peskov r...,1,russian presidenti spokesman dmitri peskov rev...
1,Ahmed Omar Khaled NajehUkrainian Foreign Minis...,1,ahm omar khale najehukrainian foreign minist d...
2,"The Jordanian Seismological Observatory, Ghass...",1,jordanian seismolog observatori ghassan sweida...
3,Ahmed Omar Khaled NajehThe Russian Deputy Fore...,1,ahm omar khale najehth russian deputi foreign ...
4,Ahmed Omar Khaled Nageh The Russian presidenti...,1,ahm omar khale nageh russian presidenti spokes...


# Features and Labels

In [33]:
X = df["cleaned_text"]
y = df["target"]

print(y.unique())
print(y.value_counts())

[1 0]
target
0    5268
1    4862
Name: count, dtype: int64


# TF-IDF Vectorization

In [36]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(X)

print(X.shape)

(10130, 5000)


# Train-Test Split

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(y_train.value_counts())
print(y_test.value_counts())

target
0    4214
1    3890
Name: count, dtype: int64
target
0    1054
1     972
Name: count, dtype: int64


# Train Model

In [42]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


# Prediction

In [45]:
y_pred = model.predict(X_test)

# Accuracy

In [48]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9595261599210266


# Classification Report

In [51]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      1054
           1       0.97      0.94      0.96       972

    accuracy                           0.96      2026
   macro avg       0.96      0.96      0.96      2026
weighted avg       0.96      0.96      0.96      2026



# Save Model

In [54]:
joblib.dump(model, "fake_news_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model Saved Successfully")

Model Saved Successfully


# Load saved model

In [58]:
import joblib

model = joblib.load("fake_news_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Live prediction

In [62]:
news = input("Enter News Text: ")

news_vector = vectorizer.transform([news])

prediction = model.predict(news_vector)

if prediction[0] == 1:
    print("Fake News")
else:
    print("Real News")

Enter News Text:  "Western media is becoming unhinged as its anti-Russia propaganda struggles to keep a hold on its consumers. Two recent examples provide evidence.Pro-peace conspiracy emanating from MoscowOn August 28, the New York Times published an article by its Moscow bureau chief about the troubling news (from the Times‘ viewpoint) that the people of Sweden are not happy with their government’s wish to join up with the NATO military alliance.The ruling elites in Sweden and Finland have been quietly pushing for NATO membership for years. In May, the Swedish government pushed through the Riksdag a proposal for a ‘cooperation agreement’ with NATO, allowing it freer access to Swedish territory for transit and training. Finland already has such an agreement in place. In July, government leaders of the two countries proudly joined the NATO summit dinner in Warsaw.But as a Reuters report at the time of the Warsaw summit explained, “An SvD/SIFO opinion poll showed 49 per cent of Swedes o

Fake News
